In [1]:

from pyspark.sql import functions as f
from delta.tables import DeltaTable

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 3, Finished, Available, Finished, False)

In [2]:
df_flights = spark.read.table("cleand_flights")

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 4, Finished, Available, Finished, False)

## **Dim Airline**

In [3]:
df_airline = df_flights.select(
    f.col("airline_iata"),
    f.col("airline_name")
).distinct().filter("airline_iata is not null")

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 5, Finished, Available, Finished, False)

In [4]:
if not spark.catalog.tableExists("Dim_Airline"):
    df_airline = df_airline.withColumn("airline_key",f.expr("uuid()"))
    df_airline.write.format("delta").saveAsTable("Dim_Airline")


else:
    df_airline = df_airline.withColumn("airline_key",f.expr("uuid()"))
    df_airline = df_airline.drop_duplicates(["airline_iata"])
    airline_prev = DeltaTable.forName(spark,"Dim_Airline")
    airline_prev.alias("prev")\
    .merge(df_airline.alias("new"),"new.airline_iata = prev.airline_iata")\
    .whenMatchedUpdate(set={
        "airline_name":"new.airline_name",
    })\
    .whenNotMatchedInsert(values={
        "airline_key":"new.airline_key",
        "airline_iata":"new.airline_iata",
        "airline_name":"new.airline_name"
    }).execute()

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 6, Finished, Available, Finished, False)

## **Dim Airports**

In [5]:
df_dep_airport = df_flights.select(
    f.col("departure_airport").alias("airport_name"),
    f.col("departure_timezone").alias("airport_timezone"),
    f.col("departure_iata").alias("airport_iata")
)


df_arrival_airport = df_flights.select(
    f.col("arrival_airport").alias("airport_name"),
    f.col("arrival_timezone").alias("airport_timezone"),
    f.col("arrival_iata").alias("airport_iata")
)









StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 7, Finished, Available, Finished, False)

In [6]:
df_airports = df_dep_airport.union(df_arrival_airport).distinct().filter("airport_iata is not null")

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 8, Finished, Available, Finished, False)

In [7]:
if not spark.catalog.tableExists("Dim_Airports"):
    df_airports = df_airports.withColumn("airport_key",f.expr("uuid()"))
    df_airports.write.format("delta").saveAsTable("Dim_Airports")


else:
    df_airports = df_airports.withColumn("airport_key",f.expr("uuid()"))
    df_airports = df_airports.dropDuplicates(["airport_iata"])
    prev_airport = DeltaTable.forName(spark,"Dim_Airports")
    prev_airport.alias("prev").merge(
     df_airports.alias("new"), "prev.airport_iata = new.airport_iata"   
    ).whenMatchedUpdate(set={
        "airport_name":"new.airport_name",
        "airport_timezone":"new.airport_timezone"
    })\
    .whenNotMatchedInsert(values={
        "airport_key":"new.airport_key",
        "airport_name":"new.airport_name",
        "airport_iata":"new.airport_iata",
        "airport_timezone":"new.airport_timezone"
    }).execute()

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 9, Finished, Available, Finished, False)

## **Dim Date**

In [8]:
df_date = df_flights.select(f.to_date(f.col("flight_date")).alias("Date")).distinct().filter("Date is not null")

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 10, Finished, Available, Finished, False)

In [9]:
df_dim_date = df_date.select(
    f.date_format(f.col("Date"),"yyyyMMdd").cast("int").alias("date_key"),
    f.col("Date").alias("full_date"),
    f.year(f.col("Date")).alias("year"),
    f.month(f.col("Date")).alias("month"),
    f.date_format(f.col("Date"),"MMMM").alias("month_name"),
    f.dayofmonth(f.col("Date")).alias("day_of_month"),
    f.date_format(f.col("Date"),"EEEE").alias("day_name"),
    f.weekofyear(f.col("Date")).alias("week_of_year"),
    (f.when(f.date_format(f.col("Date"),"EEEE").isin("Friday"),"weekend").otherwise("weekday")).alias("is_weekend")
).distinct()

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 11, Finished, Available, Finished, False)

In [10]:
if not spark.catalog.tableExists("Dim_Date"):
    df_dim_date.write.format("delta").saveAsTable("Dim_Date")
else:
    prev_date = DeltaTable.forName(spark,"Dim_Date")
    prev_date.alias("prev").merge(
        df_dim_date.alias("new"),"prev.date_key= new.date_key"
    ).whenMatchedUpdate(set={
        "full_date":"new.full_date",
        "year":"new.year",
        "month":"new.month",
        "month_name":"new.month_name",
        "day_of_month":"new.day_of_month",
        "day_name":"new.day_name",
        "week_of_year":"new.week_of_year",
        "is_weekend":"new.is_weekend"
    }).whenNotMatchedInsert(values={
        "full_date":"new.full_date",
        "year":"new.year",
        "month":"new.month",
        "month_name":"new.month_name",
        "day_of_month":"new.day_of_month",
        "day_name":"new.day_name",
        "week_of_year":"new.week_of_year",
        "is_weekend":"new.is_weekend"        
    }).execute()

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 12, Finished, Available, Finished, False)

## **Fact Table**

In [11]:
dim_airline = spark.read.table("dim_airline")
dim_airport= spark.read.table("dim_airports")

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 13, Finished, Available, Finished, False)

In [12]:
df_fact= df_flights.alias("flight").join(dim_airline.alias("airline"),f.col("flight.airline_iata") == f.col("airline.airline_iata"),"left")\
.join(dim_airport.alias("dep"),f.col("flight.departure_iata")== f.col("dep.airport_iata"),"left")\
.join(dim_airport.alias("arr"),f.col("flight.arrival_iata")== f.col("arr.airport_iata"),"left")

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 14, Finished, Available, Finished, False)

In [13]:
df_fact = df_fact.select(
    f.col("flight_id"),
    f.date_format(f.col("flight_date"),"yyyyMMdd").cast("int").alias("date_key"),
    f.col("airline.airline_key"),
    f.col("dep.airport_key").alias("departure_airport_key"),
    f.col("arr.airport_key").alias("arrival_airport_key"),
    f.col("flight.flight_status"),
    f.col("flight.flight_number"),
    f.col("flight.flight_iata"),
    f.col("flight.hour"),
    f.col("flight.duration"),
    f.col("flight.time_buketing"),
    f.col("flight.flight_type")
).dropDuplicates(["flight_id"])

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 15, Finished, Available, Finished, False)

In [14]:
display(df_fact.limit(5))

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5abe2791-c9b0-4e51-9c76-49fd13bbbccf)

In [18]:
if not spark.catalog.tableExists("fact_flights"):
    df_fact.write.format("delta").saveAsTable("fact_flights")
else:
    prev_fact = DeltaTable.forName(spark,"fact_flights")
    prev_fact.alias("prev").merge(
        df_fact.alias("new"),
        f.col("prev.flight_id")==f.col("new.flight_id")
    ).whenMatchedUpdate(set={
        "date_key":"new.date_key",
        "flight_status":"new.flight_status",
        "departure_airport_key":"new.departure_airport_key",
        "arrival_airport_key":"new.arrival_airport_key",
        "flight_number":"new.flight_number",
        "flight_iata":"new.flight_iata",
        "duration":"new.duration",
        "time_buketing":"new.time_buketing",
        "flight_type":"new.flight_type"
    }).whenNotMatchedInsert(values={
        "flight_id":"new.flight_id",
        "date_key":"new.date_key",
        "flight_status":"new.flight_status",
        "departure_airport_key":"new.departure_airport_key",
        "arrival_airport_key":"new.arrival_airport_key",
        "flight_number":"new.flight_number",
        "flight_iata":"new.flight_iata",
        "duration":"new.duration",
        "time_buketing":"new.time_buketing",
        "flight_type":"new.flight_type"  
    }).execute()

StatementMeta(, d7f41fab-d604-44f9-bcab-c53a37ffa75b, 20, Finished, Available, Finished, False)